# From BED files to GPU kernels and PCA

Read SNP data onto the GPU, build a standardized kernel, and plot the first two principal components.

This requires CuPy, Matplotlib, an NVIDIA GPU, and PySnpTools 1.1.0b1 or later (with CPU-only bed-reader). See the [GPU guide](../source/gpu.rst) for setup.


In [ ]:
import os

# Default precision for this notebook; explicit dtype arguments can override it.
os.environ["ARRAY_DTYPE"] = "float32"

import cupy as cp
from pysnptools.util.gpu import has_gpu_support

if not has_gpu_support():
    raise RuntimeError("CuPy and a usable CUDA device are required")

device = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print("GPU:", device["name"].decode())


## Specify the sample data

In [ ]:
from pysnptools.snpreader import Bed
from pysnptools.util import example_file

# Download and cache the same toy data used by other PySnpTools examples.
bed_path = example_file("pysnptools/examples/toydata.*", "*.bed")
bed = Bed(bed_path, count_A1=True)
print(f"{bed.iid_count} samples × {bed.sid_count} SNPs")


## Read onto the GPU


In [ ]:

# xp=cp selects CuPy; use xp="numpy" for a CPU read.
# float32 arithmetic is usually much faster than float64 on consumer GPUs.
# This read uses the ARRAY_DTYPE setting from the first cell.
snps = bed.read(xp=cp)
print(type(snps.val), snps.val.shape, snps.val.dtype)
print("SNP values:", "on GPU:", f"{isinstance(snps.val, cp.ndarray)}")
# SNP values are on the GPU; sample and SNP metadata stay on the CPU.


## Standardize and build a kernel


In [ ]:
from pysnptools.standardizer import Unit

# Unit centers and scales each SNP, then replaces missing values with zero.
# The kernel is the standardized SNP matrix multiplied by its transpose.
# Both operations stay on the GPU and use ARRAY_DTYPE.
kernel = snps.read_kernel(Unit(), xp=cp)
print(type(kernel.val), kernel.val.shape, kernel.val.dtype)
print("Kernel:", "on GPU:", f"{isinstance(kernel.val, cp.ndarray)}")


## Run PCA and plot


In [ ]:
# CuPy performs the eigendecomposition on the GPU.
eigenvalues, eigenvectors = cp.linalg.eigh(kernel.val)
# eigh returns eigenvalues in ascending order; select the largest two.
coordinates = eigenvectors[:, [-1, -2]] * cp.sqrt(eigenvalues[[-1, -2]])

# New CuPy results can have a different owner; they still live on the GPU.
print("PCA coordinates:", "on GPU:", f"{isinstance(coordinates, cp.ndarray)}")


In [ ]:
import matplotlib.pyplot as plt

# Copy only the two plotting coordinates per sample back to the CPU.
coordinates_cpu = cp.asnumpy(coordinates)
print("Plotting coordinates:", "on GPU:", f"{isinstance(coordinates_cpu, cp.ndarray)}")
figure, axes = plt.subplots(figsize=(6, 4))
axes.scatter(coordinates_cpu[:, 0], coordinates_cpu[:, 1])
axes.set(xlabel="PC1", ylabel="PC2", title="GPU kernel PCA on the toy dataset")
figure.tight_layout()
plt.show()
# Eigenvector signs are arbitrary; the plot may be reflected on another run.


## Compare kernel creation times


In [ ]:
# High-resolution elapsed-time clock, used only for this benchmark.
from time import perf_counter

import numpy as np
from pysnptools.standardizer import Identity

# Compare Unit standardization + Z @ Z.T, using the same data and the notebook's ARRAY_DTYPE setting.
# File reads and case setup are outside the timings; restoring inputs is included.
cell_started = perf_counter()
cpu = bed.read(xp=np)
original_gpu = cp.array(cpu.val)
# Fresh in-memory copies use the chosen NumPy or CuPy allocator.
cases = [("NumPy", np, cpu.read(xp=np)), ("CuPy", cp, cpu.read(xp=cp))]


def time_batch(data, xp, repetitions):
    if xp is cp:
        # Timing only: exclude earlier GPU work from this batch.
        cp.cuda.runtime.deviceSynchronize()
    started = perf_counter()
    for repetition in range(repetitions):
        # Standardization modifies values, so restore the original SNP values.
        xp.copyto(data.val, cpu.val if xp is np else original_gpu)
        data.standardize(Unit())
        # Already standardized: Identity avoids standardizing a second time.
        product = data.read_kernel(Identity(), xp=xp)
        del product
    if xp is cp:
        # Timing only: wait once for the batch, rather than after every operation.
        # Normal API use handles its own coordination for correct results.
        cp.cuda.runtime.deviceSynchronize()
    return perf_counter() - started


# Aim for ~0.8 seconds per backend, keeping the whole cell around 3–4 seconds.
# Warm up first, then estimate how many repetitions each backend needs.
results = []
for label, xp, data in cases:
    time_batch(data, xp, 1)
    seconds_per_kernel = time_batch(data, xp, 2) / 2
    repetitions = max(1, int(0.8 / seconds_per_kernel))
    duration = time_batch(data, xp, repetitions)
    results.append((label, duration * 1000 / repetitions,
                    isinstance(data.val, cp.ndarray)))

# Compare the time for the same work, independent of calibration batch sizes.
cpu_milliseconds = results[0][1]
print("Kernel creation: Unit standardization + matrix multiplication (including input reset)")
print(f"{'Backend':<12} {'ms/kernel':>10} {'Speedup vs CPU':>15} {'On GPU':>8}")
for label, milliseconds, on_gpu in results:
    speedup = f"{cpu_milliseconds / milliseconds:.1f}x"
    print(f"{label:<12} {milliseconds:>10.2f} {speedup:>15} "
          f"{'yes' if on_gpu else 'no':>8}")
print(f"Whole timing cell: {perf_counter() - cell_started:.2f} s")

del cases, data, cpu, original_gpu
